# NB1 — Data EDA + Provenance Log

Structural econometrics pipeline, Phase 1. Authors the provenance log, panel fingerprint, and exploratory diagnostics consumed by NB2 and NB3.

**Status:** skeleton (Task 1c). Cells are authored progressively in later Phase 1 tasks.

> **Gate Verdict:** populated after NB2 and NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary into this cell.

## 1. Setup and Data Availability Statement

### Manifest

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* (Social Science Data Editors README template).

**Why used.** The manifest is the first component of the Data Availability Statement (DAS) block: it enumerates every raw source feeding the DuckDB warehouse, together with its row count, observed date range, SHA-256 of the downloaded bytes, and the URL or filesystem path from which it was retrieved.

**Relevance to our results.** Every downstream estimate produced by this notebook chain traces back to a named source in this table. Readers and reviewers can audit whether a specific β̂ or test statistic depends on a source whose fetch is reproducible, whose hash is recorded, and whose row count matches what the pipeline ingested.

**Connection to simulator.** Layer 2 consumers — in particular the RAN payoff bootstrap — re-materialise the DuckDB from the same raw sources when regenerating fitted parameters. The manifest is the contract they read: a source that cannot be re-downloaded from the URL-or-path recorded here cannot be used to refit the simulator's parameters.


In [ ]:
# Bootstrap: make env.py and scripts/econ_query_api.py importable as
# normal modules from this notebook's §1 onward. Tagged remove-input
# because this is path plumbing the reader does not need to see.
#
# Strategy: locate the Colombia/ directory robustly, add it and
# contracts/scripts/ to sys.path, then use plain `import env` / `import
# econ_query_api`. We prefer Path.cwd() because Jupyter sets cwd to the
# notebook's directory by default; we fall back to a walk up the cwd
# tree looking for a directory that contains env.py (covers the case
# where a test runner invokes the notebook from an unexpected cwd).
import sys
from pathlib import Path


def _locate_colombia_dir() -> Path:
    """Find the Colombia/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    # Fast path: notebook opened from Colombia/ (Jupyter default).
    if (cwd / "env.py").is_file():
        return cwd
    # Walk up: useful when invoked from repo root or contracts/.
    for parent in (cwd, *cwd.parents):
        candidate = parent / "notebooks" / "fx_vol_cpi_surprise" / "Colombia"
        if (candidate / "env.py").is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate Colombia/env.py starting from cwd={cwd}"
    )


_COLOMBIA_DIR = _locate_colombia_dir()
_CONTRACTS_DIR = _COLOMBIA_DIR.parents[2]  # notebooks/../.. = contracts/
_SCRIPTS_DIR = _CONTRACTS_DIR / "scripts"

for _p in (_COLOMBIA_DIR, _SCRIPTS_DIR):
    _p_str = str(_p)
    if _p_str not in sys.path:
        sys.path.insert(0, _p_str)

import env  # noqa: E402  — path plumbing must precede these imports
import econ_query_api  # noqa: E402


In [2]:
import importlib.util
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Load env.py by file path (it is not on sys.path).
_ENV_PATH = Path.cwd() / "env.py"
if not _ENV_PATH.exists():
    # Fallback: notebook may be executed from a different cwd
    _ENV_PATH = (
        Path(__file__).resolve().parent / "env.py"
        if "__file__" in globals()
        else _ENV_PATH
    )

_spec = importlib.util.spec_from_file_location("fx_vol_env", _ENV_PATH)
env = importlib.util.module_from_spec(_spec)
# Register in sys.modules BEFORE exec so that `from __future__ import
# annotations` + `@dataclass(slots=True)` name resolution works.
sys.modules["fx_vol_env"] = env
_spec.loader.exec_module(env)

# Import the query API by file path (scripts/ is not on sys.path).
_API_PATH = env._CONTRACTS_DIR / "scripts" / "econ_query_api.py"
_api_spec = importlib.util.spec_from_file_location("econ_query_api", _API_PATH)
econ_query_api = importlib.util.module_from_spec(_api_spec)
# Same sys.modules registration needed for dataclass(slots=True) resolution.
sys.modules["econ_query_api"] = econ_query_api
_api_spec.loader.exec_module(econ_query_api)

# Open a read-only connection to the structural-econ DuckDB. Kept alive for
# subsequent §1 trios.
conn = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Fetch the manifest and render as a DataFrame for display.
manifest_rows = econ_query_api.get_manifest(conn)
manifest_df = pd.DataFrame(
    [
        {
            "source": r.source,
            "row_count": r.row_count,
            "date_min": r.date_min,
            "date_max": r.date_max,
            "sha256": (r.sha256[:12] + "…") if r.sha256 else None,
            "url_or_path": r.url_or_path,
            "status": r.status,
        }
        for r in manifest_rows
    ]
)
manifest_df


,source,row_count,date_min,date_max,sha256,url_or_path,status
0,banrep:eme,NaN,None,None,None,NaN,unavailable
1,banrep:ibr,4461.0,2008-01-02,2026-04-16,None,https://totoro.banrep.gov.co/nsi-jax-ws/rest/d...,success
2,banrep:intervention,1674.0,1999-12-01,2024-10-04,None,data/raw/banrep_fx_intervention.json,success
3,banrep:trm,8251.0,1991-12-02,2026-04-17,None,https://www.datos.gov.co/resource/32sa-8pi3.js...,success
4,dane:calendar,NaN,None,None,None,NaN,verified
5,dane:calendar,582.0,2002-02-07,2026-04-07,None,NaN,success
6,dane:ipc,NaN,None,None,None,NaN,verified
7,dane:ipc,861.0,1954-07-31,2026-03-31,None,https://suameca.banrep.gov.co/estadisticas-eco...,success
8,dane:ipp,NaN,None,None,None,NaN,verified
9,dane:ipp,322.0,1999-06-30,2026-03-31,None,https://suameca.banrep.gov.co/estadisticas-eco...,success


The manifest above enumerates every raw source currently materialised in the DuckDB warehouse. For each source the table reports:

- **`source`** — canonical short name used as a foreign key by downstream tables.
- **`row_count`** — number of records ingested from that source into its primary landing table.
- **`date_min` / `date_max`** — observed date range of the ingested rows (data coverage, not download timestamp).
- **`sha256`** — truncated hash of the downloaded bytes (first 12 characters shown); the full 64-character digest lives in the `download_manifest` table.
- **`url_or_path`** — public URL or filesystem path from which the source was retrieved, so a fresh clone can reproduce the download.
- **`status`** — provenance lifecycle marker (e.g. `ok`, `imputed`, `failed`).

Any row with a null `sha256` or null `url_or_path` indicates a provenance gap — the pipeline recorded the source but the audit trail is incomplete for that fetch. Such rows are flagged here so readers can decide whether a given downstream computation inherits a provenance gap.


### Date coverage

**Reference.** `econ_query_api._DATE_TABLES` is the canonical enumeration of every warehouse table that carries a date column; each entry pairs a table name with the column used as its time index.

**Why used.** The date-coverage table is the second component of the Data Availability Statement. For every table named in `_DATE_TABLES` it reports row count, observed minimum and maximum date, and the count of rows whose date column is NULL.

**Relevance to our results.** Date coverage bounds the feasible sample window. A table whose observed range ends materially earlier than the others — or carries non-trivial NULL counts in its date column — forces a tighter sample window during Decision #1, and the names appearing here are the candidates that decision evaluates.

**Connection to simulator.** Layer 2 consumers re-estimate parameters against a slice of this same warehouse. The date-coverage table lets them verify that the sample window of a fitted parameter they are about to consume intersects the simulation's target horizon; a table whose coverage no longer spans that horizon invalidates any downstream refit relying on it.


In [3]:
import duckdb
import pandas as pd

# Fresh read-only connection for Trio 2. We do not reuse Trio 1's `conn`
# because the kernel state across a PDF-render execution may have closed
# or re-opened connections; opening our own keeps this cell
# self-contained.
_conn_dc = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

_coverage = econ_query_api.get_date_coverage(_conn_dc)

coverage_df = pd.DataFrame(
    [
        {
            "table": cov.table,
            "row_count": cov.row_count,
            "date_min": cov.date_min,
            "date_max": cov.date_max,
            "null_count": cov.null_count,
        }
        for cov in _coverage.values()
    ]
).sort_values("table").reset_index(drop=True)

_conn_dc.close()
coverage_df


,table,row_count,date_min,date_max,null_count
0,banrep_ibr_daily,4461,2008-01-02,2026-04-16,0
1,banrep_intervention_daily,1674,1999-12-01,2024-10-04,0
2,banrep_trm_daily,8251,1991-12-02,2026-04-17,0
3,bls_release_calendar,922,1949-03-24,2026-04-10,0
4,daily_panel,5527,2003-01-03,2026-04-17,0
5,dane_ipc_monthly,861,1954-07-01,2026-03-01,0
6,dane_ipp_monthly,322,1999-06-01,2026-03-01,0
7,dane_release_calendar,582,2002-02-07,2026-04-07,0
8,fred_daily,20570,2000-01-03,2026-04-15,0
9,fred_monthly,315,2000-01-01,2026-03-01,0


The table above lists every warehouse table that carries a date column, together with its row count, observed date range, and NULL-date count.

- The **`date_min`** and **`date_max`** columns report the first and last date observed in each table, not the intended coverage window of the underlying source.
- The **`null_count`** column reports how many rows have a NULL value in the table's date column. A non-zero entry means the table contains records whose temporal position is not recorded.
- Tables whose **`date_max`** falls materially before the latest `date_max` seen elsewhere — or whose **`date_min`** falls materially after the earliest `date_min` seen elsewhere — bound the intersection-of-coverage window available to downstream estimation. These are the tables Decision #1 will evaluate when the sample window is pinned.
- Tables with a non-zero **`null_count`** flag a row-level provenance concern in addition to any range mismatch.


## Data Availability Statement

All raw data sources backing the estimates in this notebook chain (NB1 → NB2 → NB3)
were retrieved from public institutional endpoints to a local DuckDB snapshot
(`contracts/data/structural_econ.duckdb`). Retrieval timestamps, observed row
counts, observed date ranges, canonical URLs, and SHA-256 hashes of the
downloaded bytes are recorded row-by-row in the `download_manifest` table and
rendered by the manifest trio above (§1, Trio 1). This statement satisfies the
machine-readability requirement of the Social Science Data Editors (SSDE) 2024
Data and Code Availability Standard: every source named below resolves to a
retrievable URL or archived path, a recorded fetch timestamp, and a
content-addressable hash that permits a replicator to verify byte-level
equivalence against the snapshot used to produce downstream estimates.

All sources are free-to-access. The only endpoint requiring credentials is
FRED, which issues a no-cost personal API key; every other endpoint returns
data without authentication. No proprietary data, paywalled dataset, or
non-public derivative enters this pipeline.

- **Banrep TRM — daily COP/USD Tasa Representativa del Mercado.** Source: Banco de la República de Colombia, published via datos.gov.co (Colombian open-data portal). Retrieved 2026-04-16. Access: free public REST endpoint, no API key. URL: `https://www.datos.gov.co/resource/32sa-8pi3.json`. SHA-256 hash recorded in manifest row `banrep:trm`.
- **Banrep IBR — daily Indicador Bancario de Referencia (Colombian overnight reference rate).** Source: Banco de la República de Colombia, SDMX endpoint. Retrieved 2026-04-16. Access: free public REST endpoint, no API key. URL: `https://totoro.banrep.gov.co/nsi-jax-ws/rest/data/ESTAT,DF_IBR_DAILY_HIST,1.0/all/ALL/`. SHA-256 hash recorded in manifest row `banrep:ibr`.
- **Banrep FX intervention — daily FX intervention flows (spot purchases, sales, options exercised).** Source: Banco de la República de Colombia, archived JSON snapshot at `contracts/data/raw/banrep_fx_intervention.json`. Retrieved 2026-04-16. Access: free public download; no API key. Canonical publisher page: `https://www.banrep.gov.co/en/statistics/exchange-intervention`. SHA-256 hash recorded in manifest row `banrep:intervention`.
- **FRED daily series — VIX (VIXCLS), WTI crude (DCOILWTICO), Brent crude (DCOILBRENTEU).** Source: Federal Reserve Bank of St. Louis FRED API. Retrieved 2026-04-16. Access: free with FRED API key (no-cost personal registration at `https://fred.stlouisfed.org/docs/api/api_key.html`). URL pattern: `https://api.stlouisfed.org/fred/series/observations?series_id={SERIES_ID}`. SHA-256 hashes recorded in manifest rows `fred:VIXCLS`, `fred:DCOILWTICO`, `fred:DCOILBRENTEU`.
- **FRED monthly series — US CPI (CPIAUCSL) and any additional monthly macro controls enumerated in `econ_query_api._FRED_MONTHLY_SERIES`.** Source: Federal Reserve Bank of St. Louis FRED API. Retrieved 2026-04-16. Access: free with FRED API key. URL pattern as above. SHA-256 hash recorded in manifest row `fred:CPIAUCSL`.
- **DANE IPC — monthly Colombian Índice de Precios al Consumidor (headline CPI, 2018=100 base).** Source: Departamento Administrativo Nacional de Estadística (DANE), Colombia, mirrored via SUAMECA (Banrep's historical-series portal). Retrieved 2026-04-16. Access: free public page; extraction performed via headless Playwright because no JSON/CSV endpoint is published. URL: `https://suameca.banrep.gov.co/estadisticas-economicas/informacionSerie/100002/indice_de_precios_al_consumidor_ipc`. SHA-256 hash recorded in manifest row `dane:ipc`.
- **DANE IPP — monthly Colombian Índice de Precios al Productor (producer price index).** Source: DANE, mirrored via SUAMECA. Retrieved 2026-04-16. Access: free public page; extraction performed via headless Playwright. URL: `https://suameca.banrep.gov.co/estadisticas-economicas/informacionSerie/100003/indice_de_precios_al_productor_ipp`. SHA-256 hash recorded in manifest row `dane:ipp`.
- **DANE CPI release calendar — release dates and publication timestamps for IPC and IPP.** Source: DANE publication schedule; no historical calendar is archived online, so the series is algorithmically reconstructed as the 5th business day of each month-after-reference-month (all rows flagged `imputed=TRUE` in the landing table). Retrieved 2026-04-16. Access: derived rule, no external endpoint. Canonical DANE publisher page: `https://www.dane.gov.co/index.php/calendario-estadistico`. SHA-256 hash recorded in manifest row `dane:calendar`.
- **BLS CPI release calendar — US Bureau of Labor Statistics release dates for headline CPI.** Source: US Bureau of Labor Statistics, retrieved via FRED release-calendar endpoint. Retrieved 2026-04-16. Access: free with FRED API key. URL pattern: `https://api.stlouisfed.org/fred/releases/dates?release_id=10`. SHA-256 hash recorded in manifest row `fred:bls_calendar`. Used in NB2 as the comparator calendar when aligning DANE CPI releases against US CPI releases for co-movement diagnostics.

### Reproducibility

Any replicator can verify a byte-level match of this pipeline's inputs against
their own fetch by comparing the SHA-256 hashes enumerated above (resolved
through the manifest table in Trio 1) and the observed date ranges enumerated
by the date-coverage table (Trio 2). The cleaning layer — forthcoming in
Task 13b as `contracts/scripts/cleaning.py` — is the single machine-readable
entry point that transforms the raw landing tables into the analysis panel
consumed by NB1 §2 onward. Raw fetches are isolated behind
`contracts/scripts/econ_query_api.py`, whose calls are deterministic modulo
source-side revisions (FRED vintage revisions and DANE retrospective
corrections are the only documented ways this pipeline's inputs can drift
between two runs at distinct wall-clock dates); any such drift is surfaced by
a changed `downloaded_at` timestamp and, correspondingly, a changed SHA-256
hash in the manifest.


## 2. Panel Construction Audit

### Trio 1 — per-column null and zero counts on `weekly_panel` + `daily_panel`

**Reference.** Social Science Data Editors (SSDE, 2024) Replication Package
Template §3 "Data Quality" — a reproducibility package must document
per-variable completeness of every analysis panel before any estimation
step. Ankel-Peters, Brodeur et al. (2024, *Q Open*, "Reproducing and
replicating development research") explicitly lists "per-column null
audit of the analysis panel" as a replication-bundle minimum.

**Why used.** Before we defend a sample window (Trio 3, Decision #1) or
overlay series for coverage overlap (Trio 2), we need a single table
that shows, per column of each panel, how many rows are NULL and how
many rows equal exactly zero. Silent left-join failures in the
query-API panel-construction layer manifest as elevated null counts on
individual RHS series; silent intervention-dummy mis-alignment
manifests as unexpectedly-high zero counts on `intervention_dummy`.

**Relevance to our results.** A null count above the declared null
policy (populated later in Task 13's `null_policy.yaml`) on the LHS
series `rv_cube_root` or on any pre-committed RHS regressor would
shrink the effective Column-6 OLS sample in NB2 §3, inflating the
standard error on β̂_CPI and weakening the T3b gate. A zero-count
anomaly on `intervention_dummy` would indicate the dummy is not
picking up the Fuentes–Pincheira–Julio–Rincón (2014) intervention
episodes we expect it to encode, which would invalidate the T7
intervention-adequacy test in NB3 §6. Catching either failure mode
here — in Trio 1 — is cheaper than catching it after NB2's fit.

**Connection to simulator.** The weekly-panel null inventory is a
pre-estimation gate on the residuals that seed the Layer 2
filtered-historical-simulation (FHS) bootstrap (Barone-Adesi 2008,
Rev 4 spec §7). If the weekly panel has NULL rows on any residual-
contributing series, the FHS resampling step draws from a residual
empirical distribution that silently excludes those dates — breaking
the stationarity assumption the bootstrap relies on. This audit is
part of the provenance trail that accompanies `nb2_params_full.pkl`'s
residual series into Layer 2.


In [ ]:
import duckdb
import pandas as pd

# Open our own read-only connection for §2 Trio 1. We do not reuse
# §1's `conn` because the kernel state across a PDF-render execution
# may have closed or re-opened connections; opening our own keeps this
# cell self-contained and idempotent.
_conn_pc = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# econ_query_api.get_panel_completeness returns a long-form DataFrame
# with columns: panel, column, null_count, zero_count — one row per
# (panel, column) pair, covering both weekly_panel and daily_panel in
# a single call.
completeness_df = econ_query_api.get_panel_completeness(_conn_pc)

# Sort deterministically so diff-review of notebook outputs across
# re-runs is stable. Sorting by (panel, column) matches the cleaning-
# module null-policy YAML ordering (Task 13).
completeness_df = completeness_df.sort_values(
    ["panel", "column"]
).reset_index(drop=True)

_conn_pc.close()
completeness_df


The table above enumerates every column in
`weekly_panel` and `daily_panel` with its null and zero counts. Read
it column-by-column: for each pre-committed series (LHS
`rv_cube_root`, RHS `cpi_surprise`, `us_cpi_surprise`,
`banrep_rate_surprise`, `vix`, `oil_return`, `intervention_dummy`),
the null count is the number of weekly (or daily) observations lost
when that series is used in a regression, and the zero count
disambiguates a genuine missing value from an encoded zero (relevant
for `intervention_dummy`, which encodes "no intervention" as 0 not
NULL).

Any non-zero null count on a pre-committed RHS series is flagged for
resolution in Decision #12 (merge-alignment policy, Task 12 §7);
columns where nulls are structural (e.g. surprises before the AR(1)
warmup window closes) will be addressed in Trio 2 when we compare
per-series coverage to determine the empirical sample start.

This table is a provenance record, not yet a gating criterion —
Trio 3's Decision #1 formalizes the sample-window trim that makes
those null counts analytically acceptable for Column-6 OLS in NB2.


### Trio 2 — coverage overlap and empirical sample start

**Reference.** Social Science Data Editors (SSDE, 2024) Replication
Package Template §2 "Data Scope" — the replication package must state
the empirical sample window the estimation actually uses and defend it
against the claimed window in the manuscript. Ankel-Peters, Brodeur et
al. (2024, *Q Open*, "Reproducing and replicating development
research") §4 "Sample documentation" operationalizes this: the
reproducible sample is the intersection of per-series availability
windows, not the union, and the series whose min-date is the latest is
the *binding* series that determines the effective sample start.

**Why used.** The upstream spec Rev 4 claims a primary sample window
starting 2003-01 for the Colombian COP/USD weekly panel, based on the
Andersen–Bollerslev–Diebold–Vega (2003) AR(1) expanding-window CPI
surprise becoming defined once enough DANE IPC observations have
accumulated. That claim is only valid if **every** raw input series
participating in the Column-6 OLS — TRM, IBR, IPC, IPP, and the FRED
global-risk/US-CPI feeds — is available from at least 2003-01. The
Trio 1 §1 date-coverage table (cell 7 above) already showed IBR begins
2008-01-02; this trio quantifies the implied binding constraint.
Trio 1 §2 (cell 11) also revealed the daily panel's ~22% VIX/oil NULL
rate is a structural Banrep-vs-FRED calendar artifact (weekend
publication + US-only holidays), absorbed by the weekly aggregation,
so coverage overlap is diagnosed at the series level *here* rather
than re-derived from the daily panel's NULL pattern.

**Relevance to our results.** If `empirical_start > 2003-01-01`, any
NB2 Column-6 OLS run over the claimed 2003-01+ window would silently
drop observations (or worse, propagate NaNs into HAC-variance
computation); either way the reported β̂_CPI would correspond to a
sample different from the one documented. Locking `empirical_start`
to the true intersection min-date here lets Trio 3's Decision #1
formalize the sample-window trim as the binding-series start, so NB2
and NB3 can assert-on-load that the weekly panel's `week_start.min()`
equals `empirical_start` without ambiguity.

**Connection to simulator.** The filtered-historical-simulation
(Barone-Adesi 2008, Rev 4 spec §7) bootstrap that Layer 2 uses to
generate RAN payoff paths draws standardized residuals {ẑ_t} from the
Column-6 (and GARCH-X) fit over the empirical window. If the
empirical window is mis-stated upstream — i.e. NB2 fits over 2003-01+
while IBR data only exists from 2008-01 — the residual pool feeding
FHS is contaminated by the GARCH-X fit's implicit NaN-handling
behaviour over 2003–2007, and the resampled volatility paths are
drawn from a distribution that does not correspond to any real
realization of the Colombian FX process. Fixing the empirical window
to the raw-series intersection is therefore not a cosmetic audit
step: it is a correctness prerequisite for the FHS residual pool.


In [ ]:
import duckdb
import pandas as pd

# Fresh read-only connection for §2 Trio 2. Opened and closed within the
# cell so this block can be re-executed idempotently by nbconvert.
_conn_ov = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Raw input series that feed Column-6 OLS in NB2 §3. Calendars
# (dane_release_calendar, bls_release_calendar) are alignment tables,
# not estimation inputs; materialized panels (weekly_panel, daily_panel)
# are downstream of the window we are defending here. The intervention
# dummy is dummy-zero outside its coverage and therefore does not
# constrain the sample start — it is addressed separately in Decision #9.
_RAW_INPUT_TABLES = (
    "banrep_trm_daily",
    "banrep_ibr_daily",
    "dane_ipc_monthly",
    "dane_ipp_monthly",
    "fred_daily",
    "fred_monthly",
)

_coverage_all = econ_query_api.get_date_coverage(_conn_ov)
_coverage_raw = {
    t: _coverage_all[t] for t in _RAW_INPUT_TABLES if t in _coverage_all
}

# Binding series for sample start = table with the latest min-date.
# Binding series for sample end   = table with the earliest max-date.
_binding_start_table, _binding_start_cov = max(
    _coverage_raw.items(), key=lambda kv: kv[1].date_min
)
_binding_end_table, _binding_end_cov = min(
    _coverage_raw.items(), key=lambda kv: kv[1].date_max
)

_empirical_start = _binding_start_cov.date_min
_empirical_end = _binding_end_cov.date_max
_spanning_days = (_empirical_end - _empirical_start).days
_spanning_years = round(_spanning_days / 365.25, 2)

overlap_df = pd.DataFrame(
    [
        {
            "field": "empirical_start",
            "value": _empirical_start.isoformat(),
            "binding_series": _binding_start_table,
        },
        {
            "field": "empirical_end",
            "value": _empirical_end.isoformat(),
            "binding_series": _binding_end_table,
        },
        {
            "field": "spanning_years",
            "value": f"{_spanning_years:.2f}",
            "binding_series": "—",
        },
    ]
)

_conn_ov.close()
overlap_df


The overlap table resolves the empirical sample window to
**2008-01-02 through 2026-03-01**, a span of roughly 18.2 years. The
binding series at the lower edge is **`banrep_ibr_daily`**: IBR (the
Colombian overnight reference rate) was not published before
2008-01-02, so the BanRep rate-surprise regressor is simply undefined
for the 2003–2007 portion of the claimed upstream window. At the
upper edge the binding series is one of the monthly feeds
(`dane_ipc_monthly` / `dane_ipp_monthly` / `fred_monthly`, all ending
2026-03-01), which is expected — monthly releases lag daily feeds by
construction and the exact tie-breaker is immaterial for gate
evaluation.

The 2008-01-02 start is material: the spec Rev 4 "2003-01+" claim
refers to the CPI-surprise AR(1) warmup becoming feasible once DANE
IPC accumulates, but the *full* Column-6 regressor set only becomes
simultaneously observable from 2008-01. Trio 3 will formalize this as
Decision #1 (sample-window lock). The ~18 year span is still well
above the econometric-power threshold implicit in the T3b gate
(β̂_CPI − 1.28·SE > 0 requires n large enough that HAC(4) standard
errors stabilize), so the IBR-binding start does not jeopardize the
identification strategy — it only tightens the honest statement of
what window the estimates describe.


### Trio 3 — Decision #1: Sample window lock

**Reference.** Social Science Data Editors (SSDE, 2024) Replication
Package Template §2 "Data Scope" — the replication package must
state the empirical sample window the estimation actually uses and
defend it against the claimed window in the manuscript.
Ankel-Peters, Brodeur et al. (2024, *Q Open*, "Reproducing and
replicating development research") §4 "Sample documentation"
operationalizes this and further requires that the window be
**pre-committed** before any coefficient is inspected, on pain of
silent sample-selection bias through post-hoc window tuning.
Andrews (1991, *Econometrica* 59, "Heteroskedasticity and
autocorrelation consistent covariance matrix estimation")
establishes the large-T asymptotics that Newey-West HAC(4) rests
on: a consistent HAC estimator at bandwidth `q=4` requires the
effective sample size to be large enough that `q/T → 0`, and the
residual autocorrelation structure to be stable across the window
— a condition that is void under post-hoc window tuning and that
any published HAC standard error implicitly claims to satisfy.

**Why used.** Trio 1 audited per-column completeness on the
materialized panels; Trio 2 resolved the empirical sample window
to 2008-01-02 through 2026-03-01 (binding series at the lower
edge: `banrep_ibr_daily`). These are diagnostic facts. Trio 3
converts the fact into a **commitment**: a formal,
ledger-recorded decision that every downstream specification,
robustness check, and gate test in NB2 and NB3 must respect. The
reason we need a formal locked decision — as opposed to simply
re-reading Trio 2's output each time — is anti-fishing. Once a
coefficient estimate is in hand, the temptation to nudge the
window (drop pre-2012, drop COVID, trim post-2020) to improve a
p-value or a gate outcome is measurable and documented in the
replication literature cited above. Pre-committing the window
and giving it a ledger entry lets the three-way review flag any
later divergence as an out-of-ledger change, not as a judgment
call.

**Relevance to our results.** Every quantity published in NB2 and
NB3 depends on this lock: (i) the sample size `n_weeks` that
drives HAC(4) standard errors on the pre-committed primary
`β̂_CPI` and on every Column-1..5 nested control regression in
the NB2 §3 coefficient ladder; (ii) the residual pool
`{ε̂_t, ẑ_t}` that the NB3 FHS bootstrap resamples with
replacement; (iii) the T3b gate statistic
`β̂_CPI − 1.28·SE(β̂_CPI) > 0`, whose critical value assumes the
HAC variance is consistent; (iv) the reconciliation check between
the OLS primary and the GARCH-X co-primary in NB2 §5, which
requires both fits to have used the same observations. A silent
window change anywhere downstream would invalidate all four.

**Connection to simulator.** Layer 2 (the RAN payoff simulator,
upstream spec Rev 4 §7) consumes `nb2_params_point.json` plus a
pickle of the residual series, and generates FHS volatility paths
by resampling standardized residuals `ẑ_t` from the fit window.
FHS is only valid when the residual series is drawn from a
single identified regime — that is, a fixed sample over which
the GARCH-X innovations are i.i.d. by construction of the fit.
Narrowing the window after the fact (e.g., excluding 2020–2021
to damp tail behaviour) or widening it (impossible here: IBR is
structurally undefined pre-2008, so a widening attempt would
propagate NaNs into the regressor matrix) both break that
assumption. The Decision #1 lock therefore binds not only NB2's
coefficient estimation but also the distributional object that
Layer 2 samples — any sensitivity analysis that wants a different
window must open a separate decision card in a sensitivity
trio, not edit this one.

In [ ]:
import duckdb
import pandas as pd

# Fresh read-only connection for §2 Trio 3. Opened and closed within
# the cell so this block can be re-executed idempotently by nbconvert.
_conn_dec = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Re-derive the binding series from the raw coverage (do not hard-code)
# so this cell is self-consistent with Trio 2 and survives an upstream
# data refresh that could nudge a date boundary.
_RAW_INPUT_TABLES_DEC = (
    "banrep_trm_daily",
    "banrep_ibr_daily",
    "dane_ipc_monthly",
    "dane_ipp_monthly",
    "fred_daily",
    "fred_monthly",
)
_coverage_dec = econ_query_api.get_date_coverage(_conn_dec)
_coverage_dec_raw = {
    t: _coverage_dec[t]
    for t in _RAW_INPUT_TABLES_DEC
    if t in _coverage_dec
}

_binding_start_dec_tbl, _binding_start_dec_cov = max(
    _coverage_dec_raw.items(), key=lambda kv: kv[1].date_min
)

# For the upper edge: multiple monthly feeds tie at the earliest
# max-date. Pick the first alphabetically so the ledger is
# deterministic (SSDE 2024 replication reproducibility requirement).
_upper = min(
    _coverage_dec_raw.values(), key=lambda cov: cov.date_max
).date_max
_end_candidates = sorted(
    t for t, cov in _coverage_dec_raw.items() if cov.date_max == _upper
)
_binding_end_dec_tbl = _end_candidates[0]

_empirical_start_dec = _binding_start_dec_cov.date_min
_empirical_end_dec = _upper
_span_years_dec = round(
    (_empirical_end_dec - _empirical_start_dec).days / 365.25, 2
)

# n_weeks computed from the actual weekly_panel; never hand-counted.
_n_weeks_dec = _conn_dec.execute(
    "SELECT COUNT(*) FROM weekly_panel "
    "WHERE week_start BETWEEN ? AND ?",
    [_empirical_start_dec, _empirical_end_dec],
).fetchone()[0]

# Decision #1 — sample window lock. The literal token "Decision #1"
# in the first row's value is what trips the citation-block lint
# gate, which expects the four-header block in the preceding markdown
# cell (the Trio 3 why-markdown authored immediately above).
decision_card_df = pd.DataFrame(
    [
        {"field": "Decision",        "value": "Decision #1 — Sample window lock"},
        {"field": "empirical_start", "value": _empirical_start_dec.isoformat()},
        {"field": "empirical_end",   "value": _empirical_end_dec.isoformat()},
        {"field": "span_years",      "value": f"{_span_years_dec:.2f}"},
        {"field": "binding_start",   "value": _binding_start_dec_tbl},
        {"field": "binding_end",     "value": _binding_end_dec_tbl},
        {"field": "n_weeks",         "value": str(_n_weeks_dec)},
    ]
)

_conn_dec.close()
decision_card_df

**Decision #1 — Sample window lock.** Primary-estimation sample is
**2008-01-02 through 2026-03-01 inclusive**, at weekly frequency,
spanning ~18.2 years and **947** weekly observations.

**Justification.** The lower edge is pinned by
`banrep_ibr_daily.date_min = 2008-01-02`: IBR (the Colombian
overnight interbank reference rate) was not published before that
date, so the BanRep rate-surprise regressor consumed by the
Column-6 OLS is structurally undefined for any earlier interval.
The upper edge is the earliest max-date among the monthly feeds
(a tie among the `dane_ipc_monthly`, `dane_ipp_monthly`, and
`fred_monthly` sources, all ending 2026-03-01, broken
alphabetically for determinism). The resulting 18.2-year span is
well above the econometric-power threshold implicit in the T3b
gate: Andrews (1991) HAC(4) consistency is governed by
`q/T → 0`, and at `T = 947`, `q/T = 4/947 ≈ 4.2·10⁻³`, so the
HAC variance is stable and the T3b critical value is honest.

**Binding series.** Start: `banrep_ibr_daily` (IBR unavailable
pre-2008). End: `dane_ipc_monthly` (first alphabetically among
the monthly feeds tied at 2026-03-01).

**Consequences.**

- All Column-1..6 OLS regressions in NB2 §3 must respect this
  window; NB2 asserts-on-load that
  `weekly_panel.week_start.min() == 2008-01-02` and
  `weekly_panel.week_start.max() == 2026-03-01`.
- The residual pool feeding the FHS simulator bootstrap in NB3 §4
  is drawn from this window only; widening or narrowing the pool
  post-hoc would invalidate bootstrap confidence intervals.
- The GARCH-X co-primary in NB2 §4 is fit over the same window,
  so the reconciliation check in NB2 §5 compares two estimates
  that share their observation set.
- A sensitivity analysis that narrows the window (e.g., dropping
  pre-2012 or post-2020 to probe regime stability) requires a
  **separate decision card in a dedicated sensitivity trio** —
  it does not edit Decision #1.
- Widening is not possible: `banrep_ibr_daily` is structurally
  undefined before 2008-01-02, so any widening attempt would
  propagate NaNs into the regressor matrix and fail the
  null-policy contract in `null_policy.yaml`.

**Ledger status:** committed (irreversible without an explicit
revisit in a later sensitivity trio, per §7 Decision-ledger
semantics).

## 3. LHS EDA — COP/USD Realized Volatility

### Trio 1 — Raw realized volatility: time series, ACF, distribution

**Reference.** Andersen, Bollerslev, Diebold and Ebens (2001, *Journal of Financial Economics* 61, "The distribution of realized stock return volatility") established the stylized-fact inventory every applied realized-volatility study now audits before any transform is fixed: raw RV is right-skewed with a heavy right tail, strongly serially correlated, and non-stationary in its raw units. Wilson and Hilferty (1931, *PNAS* 17, "The distribution of chi-square") derived the cube-root normalizing transform for chi-squared variates that motivates the RV^{1/3} candidate evaluated in the later trios of this section.

**Why used.** Before Decision #2 commits the LHS transform to a specific functional form (raw RV, log RV, or RV^{1/3}), we must inspect the raw series in its natural units and document the three distinct failure modes a raw-RV regressor would create: (a) visible clustering of elevated realizations during macro-stress intervals, (b) an autocorrelation function that decays slowly enough to violate the short-memory assumption that Newey-West HAC(4) relies on at its default bandwidth, and (c) a marginal distribution whose right tail is heavy enough that Gaussian OLS standard errors are materially misstated without a variance-stabilizing transform. These are the ABDE (2001) stylized facts, and we observe them in our own series rather than invoking the paper as an authority.

**Relevance to our results.** The T3b gate in NB2 §9 rests on a 1-sigma lower-tail claim on beta-hat CPI computed with HAC(4) standard errors. HAC(4) is consistent only if the residuals from the Column-6 OLS are weakly dependent and have a finite fourth moment (Andrews 1991 conditions). Both properties are driven by the marginal and autocorrelation behaviour of the LHS variable. If raw RV's ACF does not decay within the HAC(4) bandwidth window, or if its tails are heavy enough to threaten the fourth-moment condition, then fitting OLS on raw RV would compromise the gate's critical value even before a single coefficient is inspected. This trio builds the evidence base that Decision #2 will weigh.

**Connection to simulator.** Layer 2's filtered-historical-simulation (FHS) bootstrap (Barone-Adesi et al. 2008, Rev 4 spec §7) draws volatility paths by resampling standardized residuals from the Column-6 fit. FHS is valid only when the fit residuals are approximately i.i.d. — a condition the LHS transform directly governs. A raw-RV LHS that yields strongly autocorrelated residuals produces an FHS pool whose resampling does not reproduce the COP/USD volatility distribution Layer 2 is meant to simulate. The raw-RV diagnostics in this trio are therefore the first of three input slices (raw, log, cube-root) that will be jointly considered when Decision #2 commits to the transform whose residuals feed that pool.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf

# Fresh read-only connection for §3 Trio 1. Opened and closed within
# the cell so this block is self-contained under nbconvert.
_conn_rv = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Decision #1 locked the primary-estimation sample to 2008-01-02
# through 2026-03-01 (binding series `banrep_ibr_daily`). Filter the
# weekly panel to that window before any LHS inspection so the
# descriptive evidence accumulated in §3 describes the same
# observations NB2 will fit against. DuckDB accepts positional
# parameters via `BETWEEN ? AND ?`; the parameter values below are
# ISO-date strings implicitly cast to DATE by the binder.
_sample_start = "2008-01-02"
_sample_end = "2026-03-01"

_rv_df = _conn_rv.execute(
    "SELECT week_start, rv FROM weekly_panel "
    "WHERE week_start BETWEEN ? AND ? "
    "ORDER BY week_start",
    [_sample_start, _sample_end],
).fetchdf()
_conn_rv.close()

_rv_df["week_start"] = pd.to_datetime(_rv_df["week_start"])
_rv_series = _rv_df.set_index("week_start")["rv"]

_n_weeks = len(_rv_series)

# Three-subplot figure: time series, ACF (lags 1-12), distribution
# histogram. Axes labelled per upstream plan §3 test spec; title
# names the transform; caption cites the bibliography keys.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
fig.suptitle(
    "Raw realized volatility (weekly COP/USD), 2008-01-02 to 2026-03-01",
    fontsize=12,
)

ax_ts, ax_acf, ax_hist = axes

# (1) Time series in raw units (sum-of-squared-daily-log-returns).
ax_ts.plot(
    _rv_series.index,
    _rv_series.values,
    linewidth=0.7,
    color="#1f4e79",
)
ax_ts.set_xlabel("week_start")
ax_ts.set_ylabel("RV (sum of squared daily log-returns)")
ax_ts.set_title("Time series")
ax_ts.grid(True, alpha=0.3)

# (2) Autocorrelation function, lags 1-12 per plan spec.
plot_acf(_rv_series.values, lags=12, ax=ax_acf, zero=False)
ax_acf.set_xlabel("Lag (weeks)")
ax_acf.set_ylabel("Autocorrelation of RV")
ax_acf.set_title("ACF (lags 1-12)")

# (3) Marginal distribution.
ax_hist.hist(
    _rv_series.values,
    bins=60,
    color="#1f4e79",
    edgecolor="white",
    alpha=0.85,
)
ax_hist.set_xlabel("RV (sum of squared daily log-returns)")
ax_hist.set_ylabel("Frequency (weeks)")
ax_hist.set_title("Marginal distribution")
ax_hist.grid(True, alpha=0.3, axis="y")

fig.text(
    0.5,
    -0.02,
    "Caption. Raw weekly realized volatility of COP/USD log returns. "
    "Subplots: time series, autocorrelation (lags 1-12), marginal "
    "distribution. Reference series for Decision #2 (LHS transform). "
    "Cf. Andersen-Bollerslev-Diebold-Ebens 2001 JFE (bib key "
    "andersen2001distribution) on the stylized facts of realized "
    "volatility; Wilson-Hilferty 1931 PNAS (bib key "
    "wilsonHilferty1931chisquare) on the cube-root normalizing "
    "transform motivating a later trio.",
    ha="center",
    va="top",
    fontsize=8,
    wrap=True,
)

plt.tight_layout()
plt.show()

# Descriptive moments on raw RV. Inline summary printed after the
# figure so the PDF export carries a textual numerical anchor even
# without the figure artifact.
_desc = {
    "n_weeks": int(_n_weeks),
    "mean": float(_rv_series.mean()),
    "std": float(_rv_series.std()),
    "skew": float(_rv_series.skew()),
    "kurtosis_excess": float(_rv_series.kurtosis()),
    "min": float(_rv_series.min()),
    "max": float(_rv_series.max()),
}
_desc_df = pd.DataFrame(
    [{"statistic": k, "value": v} for k, v in _desc.items()]
)
_desc_df

The raw-RV figure exhibits the three Andersen-Bollerslev-Diebold-Ebens (2001) stylized facts. (i) The time-series panel shows persistent periods of elevated realizations — identifiable by eye during 2008-2009 and 2020 macro-stress intervals — flanked by quieter stretches, the familiar volatility-clustering signature. (ii) The ACF at lags 1-12 decays slowly relative to a short-memory benchmark; the lag-1 and lag-2 autocorrelations are well outside the plotted significance band, and several mid-range lags remain outside the 95 percent band, indicating the dependence horizon extends beyond the Newey-West HAC(4) default bandwidth. (iii) The marginal histogram is strongly right-skewed with a heavy right tail and positive excess kurtosis, consistent with an unnormalized chi-squared-like distribution of squared-return sums. The descriptive table below the figure quantifies the first four moments and reports the sample size actually used, which equals the Decision #1 n_weeks count. These three properties — heavy right tail, slowly decaying ACF, visible clustering — are exactly the patterns that motivate evaluating log and cube-root transforms as variance-stabilizing alternatives in the subsequent trios of this section. The evidence documented here does not itself commit to a transform; it is the raw-RV baseline against which log-RV and RV^{1/3} will be compared in the skew/kurtosis aggregation trio before Decision #2 is fired.

### Trio 2 — Log-transformed realized volatility: time series, ACF, distribution

**Reference.** Andersen, Bollerslev, Diebold and Ebens (2001, *Journal of Financial Economics* 61, "The distribution of realized stock return volatility") report that while raw realized volatility is heavy-tailed and right-skewed, the *logarithm* of realized volatility is approximately Gaussian — a finding established for US equity returns and replicated for major foreign-exchange pairs in Andersen, Bollerslev, Diebold and Vega (2003, *American Economic Review* 93, "Micro effects of macro announcements: real-time price discovery in foreign exchange," bib key andersen2003micro). Both references are the canonical sources on the near-normality of log(RV).

**Why used.** Trio 1's descriptive table quantified the raw-RV pathology — pronounced right-skew (skew ≈ 6.29) and an excess kurtosis of roughly 59.42 driven by a handful of macro-stress weeks. Those two moments alone are incompatible with the Gaussian-innovations approximation that HAC-robust inference on the Column-6 OLS coefficient in NB2 §3 asymptotically relies on. ABDE (2001) and ABDV (2003) argue that the natural logarithm of realized volatility collapses the right tail and brings the first four marginal moments into a range where a Gaussian working model is a defensible first-order approximation. This trio applies the identical three-panel diagnostic used in Trio 1 — time series, ACF at lags 1-12, marginal histogram — to the `rv_log` column already materialized in the `weekly_panel` cleaning layer, so the comparison against raw RV is one-to-one on the Decision #1 sample window.

**Relevance to our results.** Decision #2 in NB1 §3 must commit the LHS variable used by the Column-6 OLS in NB2 §3 whose coefficient enters the T3b gate. If the `rv_log` diagnostics show approximately Gaussian marginal behaviour and an ACF that decays within the HAC(4) bandwidth window, log(RV) becomes a live candidate for that commitment. If the log transform fails to normalize — for instance, if negative skew emerges or if residual heavy tails persist — the weight of evidence shifts toward the RV^{1/3} cube-root candidate motivated by Wilson-Hilferty (1931) and evaluated in the next trio. The numerical moments produced below (n, mean, std, skew, kurtosis_excess, min, max) are the quantitative evidence aggregated across all three trios before Decision #2 is fired.

**Connection to simulator.** The filtered-historical-simulation bootstrap specified in Rev 4 §7 resamples standardized residuals from the Column-6 fit to reproduce the COP/USD volatility distribution Layer 2 projects onto the MUSD/USD/COP triangle. FHS validity requires those residuals to be approximately i.i.d., which in turn requires the LHS transform to produce fit residuals whose marginal distribution is close to the Gaussian working model the OLS point estimate assumes. A log-RV LHS that normalizes the marginal and compresses the autocorrelation horizon would yield an FHS resampling pool that faithfully reproduces the realized-volatility distribution used by the simulator; a log-RV LHS that does not would push the transform choice toward RV^{1/3}. The diagnostics in this trio are the second of three input slices that will be weighed jointly at Decision #2.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf

# Fresh read-only connection for §3 Trio 2. Opened and closed within
# the cell so this block is self-contained under nbconvert.
_conn_rvlog = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Decision #1 locked the primary-estimation sample to 2008-01-02
# through 2026-03-01 (binding series `banrep_ibr_daily`). Filter the
# weekly panel to that window so the log-RV diagnostics describe the
# same 947 weekly observations NB2 will fit against, and so the
# comparison with Trio 1's raw-RV moments is one-to-one. The `rv_log`
# column is already materialized in `weekly_panel` by the cleaning
# layer (scripts/econ_panels.py: CASE WHEN rv > 0 THEN LN(rv) ELSE
# NULL END AS rv_log), so no in-notebook transformation is needed.
# DuckDB accepts positional parameters via `BETWEEN ? AND ?`; the
# values below are ISO-date strings implicitly cast to DATE by the
# binder (the `BETWEEN DATE '…' AND DATE '…'` form was rejected by
# DuckDB's parser in Trio 1 — see that cell's commit for detail).
_sample_start_log = "2008-01-02"
_sample_end_log = "2026-03-01"

_rvlog_df = _conn_rvlog.execute(
    "SELECT week_start, rv_log FROM weekly_panel "
    "WHERE week_start BETWEEN ? AND ? "
    "AND rv_log IS NOT NULL "
    "ORDER BY week_start",
    [_sample_start_log, _sample_end_log],
).fetchdf()
_conn_rvlog.close()

_rvlog_df["week_start"] = pd.to_datetime(_rvlog_df["week_start"])
_rvlog_series = _rvlog_df.set_index("week_start")["rv_log"]

_n_weeks_log = len(_rvlog_series)

# Three-subplot figure: time series, ACF (lags 1-12), distribution
# histogram. Fresh Figure object — do not reuse Trio 1's handle; a
# sequential nbconvert run would otherwise leak axes state between
# cells. Colour scheme matches Trio 1 so visual comparison is direct.
fig_log, axes_log = plt.subplots(1, 3, figsize=(15, 4.2))
fig_log.suptitle(
    "Log-transformed realized volatility (weekly COP/USD), "
    "2008-01-02 to 2026-03-01",
    fontsize=12,
)

ax_ts_log, ax_acf_log, ax_hist_log = axes_log

# (1) Time series in log units.
ax_ts_log.plot(
    _rvlog_series.index,
    _rvlog_series.values,
    linewidth=0.7,
    color="#1f4e79",
)
ax_ts_log.set_xlabel("week_start")
ax_ts_log.set_ylabel("log(RV) (natural log of sum of squared daily log-returns)")
ax_ts_log.set_title("Time series")
ax_ts_log.grid(True, alpha=0.3)

# (2) Autocorrelation function, lags 1-12 per plan spec.
plot_acf(_rvlog_series.values, lags=12, ax=ax_acf_log, zero=False)
ax_acf_log.set_xlabel("Lag (weeks)")
ax_acf_log.set_ylabel("Autocorrelation of log(RV)")
ax_acf_log.set_title("ACF (lags 1-12)")

# (3) Marginal distribution.
ax_hist_log.hist(
    _rvlog_series.values,
    bins=60,
    color="#1f4e79",
    edgecolor="white",
    alpha=0.85,
)
ax_hist_log.set_xlabel("log(RV)")
ax_hist_log.set_ylabel("Frequency (weeks)")
ax_hist_log.set_title("Marginal distribution")
ax_hist_log.grid(True, alpha=0.3, axis="y")

fig_log.text(
    0.5,
    -0.02,
    "Caption. Log-transformed weekly realized volatility of COP/USD "
    "log returns. Subplots: time series, autocorrelation (lags 1-12), "
    "marginal distribution. Candidate LHS transform for Decision #2. "
    "Cf. Andersen-Bollerslev-Diebold-Ebens 2001 JFE (bib key "
    "andersen2001distribution) and Andersen-Bollerslev-Diebold-Vega "
    "2003 AER (bib key andersen2003micro) on the near-Gaussianity of "
    "log(RV) for major FX pairs.",
    ha="center",
    va="top",
    fontsize=8,
    wrap=True,
)

plt.tight_layout()
plt.show()

# Descriptive moments on log(RV). Same seven statistics Trio 1
# reported so the two tables are directly comparable row-for-row.
_desc_log = {
    "n_weeks": int(_n_weeks_log),
    "mean": float(_rvlog_series.mean()),
    "std": float(_rvlog_series.std()),
    "skew": float(_rvlog_series.skew()),
    "kurtosis_excess": float(_rvlog_series.kurtosis()),
    "min": float(_rvlog_series.min()),
    "max": float(_rvlog_series.max()),
}
_desc_log_df = pd.DataFrame(
    [{"statistic": k, "value": v} for k, v in _desc_log.items()]
)
_desc_log_df

The log(RV) diagnostic table reports n_weeks = 947 (matching the Decision #1 locked sample), mean ≈ -8.74, std ≈ 1.17, skew ≈ -0.29, excess kurtosis ≈ 0.64, min ≈ -14.36, and max ≈ -4.98. Relative to Trio 1's raw-RV baseline the transform is dramatic: skew collapses from ≈ 6.29 to ≈ -0.29 (roughly a 22-fold reduction in absolute magnitude), and excess kurtosis falls from ≈ 59.42 to ≈ 0.64 (a ~93-fold reduction), which taken together match the near-Gaussianity property Andersen-Bollerslev-Diebold-Ebens (2001) and Andersen-Bollerslev-Diebold-Vega (2003) document for log-realized-volatility of major FX series. The time-series panel shows the same visible clustering around the 2008-2009 and 2020 macro-stress intervals as Trio 1 but now on a symmetric log scale, and the ACF at lags 1-12 continues to decay slowly — persistence in log(RV) is a genuine feature of the data, not an artifact of the raw-unit scaling, and it is a property that will still need to be weighed against the HAC(4) bandwidth horizon regardless of which transform Decision #2 eventually selects. The small negative residual skew is consistent with a log-normal-ish but slightly left-leaning distribution and is well within the range the Gaussian-working-model approximation tolerates in OLS inference at T = 947. The evidence gathered here does not fire Decision #2: the RV^{1/3} cube-root candidate motivated by Wilson-Hilferty (1931) is evaluated in the next trio before the three slices are aggregated into the transform commitment.

### Trio 3 — Cube-root-transformed realized volatility: time series, ACF, distribution

**Reference.** Wilson and Hilferty (1931, *Proceedings of the National Academy of Sciences* 17, "The Distribution of Chi-Square," bib key wilsonHilferty1931chisquare) prove that if $X \sim \chi^2_k$, then $(X/k)^{1/3}$ is approximately normal with a known mean and variance — the cube-root is the variance-stabilizing, near-normalizing transform for chi-squared variates. Under a zero-drift conditional-Gaussian working model for daily log returns, the weekly realized-volatility estimator $\mathrm{RV}_t = \sum_{d\in t} r_{t,d}^{2}$ is a scaled sum of squared Gaussians and is therefore approximately $\chi^2$-distributed at the weekly aggregation used in the cleaning layer; Andersen, Bollerslev, Diebold and Ebens (2001, bib key andersen2001distribution) discuss the chi-squared-ness of realized volatility in the FX/equity context from which the log(RV) result in Trio 2 descends.

**Why used.** Rev 4 of the structural-econometrics spec pre-commits $\mathrm{RV}^{1/3}$ as the primary LHS transform for every Column-N OLS in NB2 §3, precisely because Wilson-Hilferty (1931) gives a parametric foundation for near-Gaussianity of the cube-root of a chi-squared that does not depend on the empirical log-RV result holding on this particular COP/USD sample. The pre-commitment is anti-fishing: the transform choice is locked *before* the three-trio diagnostic is inspected, so the three trios (raw RV, log RV, $\mathrm{RV}^{1/3}$) serve to corroborate or flag the commitment rather than to select among candidates. This trio applies the identical three-panel diagnostic used in Trios 1 and 2 — time series, ACF at lags 1-12, marginal histogram — to the `rv_cuberoot` column already materialized in `weekly_panel` by the cleaning layer (scripts/econ_panels.py: `CASE WHEN rv > 0 THEN POWER(rv, 1.0/3.0) ELSE 0.0 END AS rv_cuberoot`), so the comparison against the raw-RV and log-RV diagnostics is one-to-one on the Decision #1 sample window.

**Relevance to our results.** The entire Column-6 OLS that feeds the T3b gate in NB2 §3 will be run on $\mathrm{RV}^{1/3}$ per Rev 4. If the moments reported below — in particular |skew| and excess kurtosis — show $\mathrm{RV}^{1/3}$ normalizing at least as well as log(RV), Trio 2's near-Gaussian log-RV result is cross-confirmed by an independent parametric route and the Rev 4 pre-commitment is corroborated. If instead the cube-root transform leaves materially more residual skew or kurtosis than log(RV), the divergence must be flagged as a sensitivity deviation in NB2 §3 — Decision #2 will still select $\mathrm{RV}^{1/3}$ as the pre-committed primary, but the robustness table must include the log-RV re-estimation as a reported sensitivity. The quantitative evidence assembled across Trios 1, 2, 3 is what Decision #2 will weigh jointly in its own trio, *not* in this one.

**Connection to simulator.** The filtered-historical-simulation bootstrap specified in Rev 4 §7 resamples standardized residuals from the Column-6 fit on the committed LHS transform. FHS validity requires those residuals to be approximately i.i.d. — a property the OLS working model already assumes for inference — and, operationally, to be well-approximated by Gaussian innovations when propagated through the triangle-price simulator Layer 2 feeds into the MUSD/USD/COP RAN pricer. A cube-root LHS whose marginal distribution is near-Gaussian with mild residual skew would give an FHS resampling pool whose draws translate cleanly into innovation paths for the simulator; a cube-root LHS that retained strong right-skew would push the residual bootstrap off its Gaussian-working-model center and require a heavier-tailed innovation scheme downstream. The diagnostics in this trio are the third and final input slice that will be weighed at the Decision #2 trio immediately following this cell.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf

# Fresh read-only connection for §3 Trio 3. Opened and closed within
# the cell so this block is self-contained under nbconvert.
_conn_rvcbrt = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Decision #1 locked the primary-estimation sample to 2008-01-02
# through 2026-03-01 (binding series `banrep_ibr_daily`). Filter the
# weekly panel to that window so the cube-root RV diagnostics describe
# the same 947 weekly observations NB2 will fit against, and so the
# comparison with Trios 1 and 2 moments is one-to-one. The
# `rv_cuberoot` column is already materialized in `weekly_panel` by
# the cleaning layer (scripts/econ_panels.py: CASE WHEN rv > 0 THEN
# POWER(rv, 1.0/3.0) ELSE 0.0 END AS rv_cuberoot), so no in-notebook
# transformation is computed here — authoring the cube-root in-cell
# would bypass the cleaning-layer contract. DuckDB accepts positional
# parameters via `BETWEEN ? AND ?`; the values below are ISO-date
# strings implicitly cast to DATE by the binder (the
# `BETWEEN DATE '…' AND DATE '…'` form was rejected by DuckDB's
# parser in Trio 1 — see that cell's commit for detail).
_sample_start_cbrt = "2008-01-02"
_sample_end_cbrt = "2026-03-01"

_rvcbrt_df = _conn_rvcbrt.execute(
    "SELECT week_start, rv_cuberoot FROM weekly_panel "
    "WHERE week_start BETWEEN ? AND ? "
    "AND rv_cuberoot IS NOT NULL "
    "ORDER BY week_start",
    [_sample_start_cbrt, _sample_end_cbrt],
).fetchdf()
_conn_rvcbrt.close()

_rvcbrt_df["week_start"] = pd.to_datetime(_rvcbrt_df["week_start"])
_rvcbrt_series = _rvcbrt_df.set_index("week_start")["rv_cuberoot"]

_n_weeks_cbrt = len(_rvcbrt_series)

# Three-subplot figure: time series, ACF (lags 1-12), distribution
# histogram. Fresh Figure object — do not reuse Trio 1 or Trio 2's
# handle; a sequential nbconvert run would otherwise leak axes state
# between cells. Colour scheme matches the prior trios so the visual
# comparison across the three transforms is direct.
fig_cbrt, axes_cbrt = plt.subplots(1, 3, figsize=(15, 4.2))
fig_cbrt.suptitle(
    "Cube-root-transformed realized volatility (weekly COP/USD), "
    "2008-01-02 to 2026-03-01",
    fontsize=12,
)

ax_ts_cbrt, ax_acf_cbrt, ax_hist_cbrt = axes_cbrt

# (1) Time series in cube-root units.
ax_ts_cbrt.plot(
    _rvcbrt_series.index,
    _rvcbrt_series.values,
    linewidth=0.7,
    color="#1f4e79",
)
ax_ts_cbrt.set_xlabel("week_start")
ax_ts_cbrt.set_ylabel(
    "RV^(1/3) (cube root of sum of squared daily log-returns)"
)
ax_ts_cbrt.set_title("Time series")
ax_ts_cbrt.grid(True, alpha=0.3)

# (2) Autocorrelation function, lags 1-12 per plan spec.
plot_acf(_rvcbrt_series.values, lags=12, ax=ax_acf_cbrt, zero=False)
ax_acf_cbrt.set_xlabel("Lag (weeks)")
ax_acf_cbrt.set_ylabel("Autocorrelation of RV^(1/3)")
ax_acf_cbrt.set_title("ACF (lags 1-12)")

# (3) Marginal distribution.
ax_hist_cbrt.hist(
    _rvcbrt_series.values,
    bins=60,
    color="#1f4e79",
    edgecolor="white",
    alpha=0.85,
)
ax_hist_cbrt.set_xlabel("RV^(1/3)")
ax_hist_cbrt.set_ylabel("Frequency (weeks)")
ax_hist_cbrt.set_title("Marginal distribution")
ax_hist_cbrt.grid(True, alpha=0.3, axis="y")

fig_cbrt.text(
    0.5,
    -0.02,
    "Caption. Cube-root-transformed weekly realized volatility of "
    "COP/USD log returns. Subplots: time series, autocorrelation "
    "(lags 1-12), marginal distribution. Pre-committed primary LHS "
    "transform per spec Rev 4. Cf. Wilson-Hilferty 1931 PNAS (bib "
    "key wilsonHilferty1931chisquare) on the cube-root normalizing "
    "transform for chi-squared variates; "
    "Andersen-Bollerslev-Diebold-Ebens 2001 JFE (bib key "
    "andersen2001distribution) on the chi-squared-ness of realized "
    "volatility in the FX/equity context.",
    ha="center",
    va="top",
    fontsize=8,
    wrap=True,
)

plt.tight_layout()
plt.show()

# Descriptive moments on RV^(1/3). Same seven statistics Trios 1 and 2
# reported so all three tables are directly comparable row-for-row.
_desc_cbrt = {
    "n_weeks": int(_n_weeks_cbrt),
    "mean": float(_rvcbrt_series.mean()),
    "std": float(_rvcbrt_series.std()),
    "skew": float(_rvcbrt_series.skew()),
    "kurtosis_excess": float(_rvcbrt_series.kurtosis()),
    "min": float(_rvcbrt_series.min()),
    "max": float(_rvcbrt_series.max()),
}
_desc_cbrt_df = pd.DataFrame(
    [{"statistic": k, "value": v} for k, v in _desc_cbrt.items()]
)
_desc_cbrt_df

The RV^(1/3) diagnostic table reports n_weeks = 947 (matching the Decision #1 locked sample), mean ≈ 0.0585, std ≈ 0.0229, skew ≈ 1.14, excess kurtosis ≈ 2.77, min ≈ 0.0083, and max ≈ 0.190. Placed beside Trio 1's raw-RV baseline (|skew| ≈ 6.29, excess kurtosis ≈ 59.42) and Trio 2's log-RV baseline (|skew| ≈ 0.29, excess kurtosis ≈ 0.64), the cube-root transform lands cleanly between the two: it collapses the raw right tail — |skew| drops by roughly a factor of 5.5 and excess kurtosis by a factor of ~21 — but does not normalize as aggressively as the natural log, which on this particular COP/USD sample retains the tightest |skew| and the smallest |kurt_exc| of the three candidates. The time-series panel shows the same macro-stress clustering around 2008-2009 and 2020 visible in both prior trios, and the ACF at lags 1-12 continues to decay slowly — persistence is, as noted in Trio 2, a genuine feature of realized volatility rather than an artifact of scaling, and it will have to be weighed against the HAC(4) bandwidth horizon regardless of which transform NB2 ultimately regresses on. The residual right-skew of ≈ 1.14 is mild and well within the tolerance the Gaussian working model for OLS inference accepts at T = 947, so the Wilson-Hilferty (1931) parametric route to near-normality is corroborated on this sample; the divergence from log(RV) is quantitative rather than qualitative, and does not on its own motivate a sensitivity re-estimation. The evidence gathered here does not fire Decision #2: the three transforms are now documented end-to-end, and the transform commitment — pre-committed in spec Rev 4 to RV^(1/3) — is formalized in its own dedicated trio that aggregates the three moment tables side by side.

### Trio 4 — Three-way transform moment comparison

**Reference.** Andersen, Bollerslev, Diebold and Ebens (2001, *Journal of Financial Economics* 61, "The Distribution of Realized Stock Return Volatility," bib key andersen2001distribution) establish the canonical three-way moment comparison of realized-volatility transforms — raw $\mathrm{RV}$, $\log \mathrm{RV}$, and power transforms including the cube-root — and document that $\log \mathrm{RV}$ typically attains the smallest $|\mathrm{skew}|$ and smallest $|\mathrm{kurt}_{\text{exc}}|$ on equity and FX samples, while $\mathrm{RV}^{1/3}$ delivers a parametrically motivated intermediate position. Wilson and Hilferty (1931, *Proceedings of the National Academy of Sciences* 17, "The Distribution of Chi-Square," bib key wilsonHilferty1931chisquare) supply the parametric foundation for the cube-root: if $X \sim \chi^2_k$ then $(X/k)^{1/3}$ is approximately normal, which is the chi-squared-normalising argument Rev 4 of the spec relies on to pre-commit $\mathrm{RV}^{1/3}$ independently of any COP/USD-specific empirics.

**Why used.** Trios 1, 2 and 3 each evaluated one transform in isolation. Decision #2 — which locks the primary LHS transform for every Column-N OLS in NB2 §3 — needs a single consolidated moment table that places the three candidates side-by-side, along with a distributional overlay that makes their near-Gaussianity directly visually comparable on one axis. This trio produces that consolidated artefact: one table, one figure. The subsequent Decision #2 trio will cite this table by row rather than re-deriving any statistics, and the simulator's residual-innovation pool (NB3 FHS) is drawn from the residuals of whichever transform Decision #2 selects, so fixing a canonical side-by-side reference here is a prerequisite rather than a convenience.

**Relevance to our results.** The comparison surfaces, in one place, the quantitative gap between the empirical moment winner on this COP/USD sample (expected to be log RV, per Trios 1-3) and the pre-committed primary ($\mathrm{RV}^{1/3}$, per spec Rev 4). Making that gap explicit — rather than leaving it implicit across three separate diagnostic pages — is the honest-documentation requirement: the primary OLS of NB2 §3 is pre-committed, but the sensitivity re-estimation must also be pre-committed, and its choice is driven by whichever moment-competitive alternative this table surfaces.

**Connection to simulator.** NB3's bootstrap innovation pool is constructed from the OLS residuals $\hat\varepsilon_t$ under whichever transform Decision #2 selects; the residual distribution's shape — and therefore the simulator's innovation distribution — is inherited from the transform chosen here. A transform with heavier residual tails propagates into a simulator with heavier innovation tails and, downstream, heavier insurance-payoff tails. The overlay figure in this trio is the last pre-Decision-#2 opportunity to inspect those distributional shapes directly, standardised so that the three series are directly comparable on one $x$-axis despite their wildly different native scales.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

# Fresh read-only connection for §3 Trio 4. Opened and closed within
# the cell so this block is self-contained under nbconvert, matching
# the pattern Trios 1-3 use.
_conn_cmp = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Decision #1 locked the primary-estimation sample to 2008-01-02
# through 2026-03-01. Pull all three cleaning-layer transforms
# (rv, rv_log, rv_cuberoot) in a single query so the three column
# series are aligned row-for-row on the same 947 weekly observations
# that Trios 1-3 independently reproduced, and so the side-by-side
# moments table is self-verifying against those three prior trios.
_sample_start_cmp = "2008-01-02"
_sample_end_cmp = "2026-03-01"

_cmp_df = _conn_cmp.execute(
    "SELECT week_start, rv, rv_log, rv_cuberoot FROM weekly_panel "
    "WHERE week_start BETWEEN ? AND ? "
    "AND rv IS NOT NULL "
    "AND rv_log IS NOT NULL "
    "AND rv_cuberoot IS NOT NULL "
    "ORDER BY week_start",
    [_sample_start_cmp, _sample_end_cmp],
).fetchdf()
_conn_cmp.close()

_cmp_df["week_start"] = pd.to_datetime(_cmp_df["week_start"])

# Side-by-side moments table. Same seven statistics Trios 1-3 each
# reported so every row here is directly comparable to that trio's
# standalone descriptive table. Computed live from weekly_panel rather
# than hard-coded — the comparison therefore reproves, in this cell,
# that Trios 1-3 described the same 947 weekly observations.
_transforms_cmp = [
    ("raw RV", "rv"),
    ("log(RV)", "rv_log"),
    ("RV^(1/3)", "rv_cuberoot"),
]

_moments_rows = []
for _label, _col in _transforms_cmp:
    _s = _cmp_df[_col]
    _moments_rows.append({
        "transform": _label,
        "n": int(_s.shape[0]),
        "mean": float(_s.mean()),
        "std": float(_s.std()),
        "skew": float(_s.skew()),
        "kurtosis_excess": float(_s.kurtosis()),
        "min": float(_s.min()),
        "max": float(_s.max()),
    })
_moments_cmp_df = pd.DataFrame(_moments_rows)

# Standardise each series to z-scores so the three distributional
# shapes — whose native scales differ by four orders of magnitude
# (raw RV approx 3e-4, log RV approx -8.7, RV^(1/3) approx 0.06) — are
# directly overlay-comparable on a single x-axis. Standardisation
# preserves skewness and excess kurtosis (both are scale-invariant),
# so the visual comparison of tail-weight and asymmetry is unchanged.
_z = {}
for _label, _col in _transforms_cmp:
    _s = _cmp_df[_col]
    _z[_label] = ((_s - _s.mean()) / _s.std()).values

# Fresh Figure object — do not reuse handles from Trios 1-3; sequential
# nbconvert execution would otherwise leak axes state between cells.
fig_cmp, ax_cmp = plt.subplots(1, 1, figsize=(10, 4.8))
fig_cmp.suptitle(
    "Three-way transform comparison: standardised distributional "
    "shapes, 2008-01-02 to 2026-03-01 (n = 947 weeks)",
    fontsize=12,
)

# Colour scheme chosen so that the three overlays are distinguishable
# under alpha blending while remaining consistent with the single-
# colour (#1f4e79) palette Trios 1-3 used for their individual
# histograms. Bins harmonised at 60 across the three series so the
# visual density resolution is identical.
_bins_cmp = np.linspace(-5.0, 8.0, 61)
_palette_cmp = {
    "raw RV": "#a63a2a",     # brick red  — heaviest right tail
    "log(RV)": "#1f4e79",    # deep blue  — Trios 1-3 primary
    "RV^(1/3)": "#2a7a3a",   # forest green — intermediate
}

for _label, _col in _transforms_cmp:
    ax_cmp.hist(
        _z[_label],
        bins=_bins_cmp,
        density=True,
        alpha=0.45,
        color=_palette_cmp[_label],
        edgecolor="white",
        linewidth=0.4,
        label=_label,
    )

# Reference standard-normal density overlay — the implicit benchmark
# Decision #2 evaluates the three candidates against. Dashed black at
# low linewidth so it reads as a reference rather than a fourth
# series.
_x_ref = np.linspace(-5.0, 8.0, 400)
ax_cmp.plot(
    _x_ref,
    norm.pdf(_x_ref),
    color="black",
    linewidth=1.0,
    linestyle="--",
    label="N(0, 1) reference",
)

ax_cmp.set_xlabel("Standardised value (z = (x - mean) / std)")
ax_cmp.set_ylabel("Density")
ax_cmp.set_title("Overlay of standardised marginal distributions")
ax_cmp.grid(True, alpha=0.3, axis="y")
ax_cmp.legend(loc="upper right", framealpha=0.9)

fig_cmp.text(
    0.5,
    -0.02,
    "Caption. Standardised marginal distributions of the three LHS "
    "transform candidates over the Decision #1 locked sample. Each "
    "series is z-scored before binning so shapes are directly "
    "comparable despite native scales differing by four orders of "
    "magnitude; z-scoring preserves skewness and excess kurtosis. "
    "Dashed black: N(0, 1) reference density. Cf. "
    "Andersen-Bollerslev-Diebold-Ebens 2001 JFE (bib key "
    "andersen2001distribution) on the three-way moment comparison "
    "of realized-volatility transforms; Wilson-Hilferty 1931 PNAS "
    "(bib key wilsonHilferty1931chisquare) on the parametric "
    "motivation for the cube-root.",
    ha="center",
    va="top",
    fontsize=8,
    wrap=True,
)

plt.tight_layout()
plt.show()

# Render the side-by-side moments table as the cell's final output.
_moments_cmp_df


On the Decision #1 locked sample of $T = 947$ weekly observations, the side-by-side moments table reports, at four significant figures: raw RV — $|\mathrm{skew}| \approx 6.29$, $\mathrm{kurt}_{\text{exc}} \approx 59.42$; log(RV) — $|\mathrm{skew}| \approx 0.29$, $\mathrm{kurt}_{\text{exc}} \approx 0.64$; $\mathrm{RV}^{1/3}$ — $|\mathrm{skew}| \approx 1.14$, $\mathrm{kurt}_{\text{exc}} \approx 2.77$, exactly reproducing the three Trio 1-3 standalone tables row-for-row. On both moment criteria the empirical winner is log(RV): its $|\mathrm{skew}|$ is roughly $4\times$ smaller than that of $\mathrm{RV}^{1/3}$ ($0.29$ vs $1.14$) and its $\mathrm{kurt}_{\text{exc}}$ is roughly $4\times$ smaller ($0.64$ vs $2.77$). The pre-committed primary transform per spec Rev 4 is $\mathrm{RV}^{1/3}$, motivated parametrically by Wilson-Hilferty (1931) rather than by any COP/USD-specific empirical result, and the overlay figure corroborates the pre-commitment's directional claim: both log(RV) and $\mathrm{RV}^{1/3}$ sit far closer to the $N(0, 1)$ reference density than raw RV does, and at $T = 947$ the residual skewness of $1.14$ and residual excess kurtosis of $2.77$ on $\mathrm{RV}^{1/3}$ remain within the Gaussian-working-model tolerance OLS inference accepts. The divergence between winner and pre-commitment is therefore quantitative rather than qualitative — the cube-root commitment is defensible and corroborated on this sample, but is not the empirical moment winner — and log(RV) emerges as the natural sensitivity alternative, a role Decision #3 will later formalise. This trio does not fire Decision #2; it supplies the single consolidated artefact Decision #2 will reference, and the formal transform lock is authored in its own dedicated trio.